<a href="https://colab.research.google.com/github/Arkasain/SUPPLY_CHAIN_PROJECT/blob/main/supply_chain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from warnings import filterwarnings
filterwarnings('ignore')


In [ ]:

##set professional color them

viridis_colors = cm.viridis(np.linspace(0,1,5))
primary_color = viridis_colors[0]
secondary_color = viridis_colors[1]
accent_color = viridis_colors[2]
danger_color ='#800000'
netural_color = viridis_colors[3]
success_color = viridis_colors[4]
custom_palette = viridis_colors

In [ ]:
df=pd.read_csv("/content/new data.csv", encoding='latin1')

In [ ]:
print(df)

In [ ]:
print(df.describe())

In [ ]:
print(df.shape)

In [ ]:
df.dtypes


In [ ]:
df['Product Status'].value_counts()

EDA PERFORM

In [ ]:
df['Benefit per order'] == df['Order Profit Per Order']


**Data Cleaning And Feature Engineering process will be done..**

In [ ]:
#DATA CLEANING
columns_to_drop = [
    'Product Description',
    'Product Image',
    'Customer Email',
    'Customer Password',
    'Customer Fname',
    'Customer Lname',
    'Customer Street',
    'Customer Zipcode',
    'Order Zipcode',
    'Latitude',
    'Longitude',
    'Order Item Cardprod Id',
    'Order Item Id',
    'Order Item Product Price',
    'Order Item Quantity',
    'Order Item Total',
    'Category Id',
    'Department Id',
    'Product Card Id',
    'Product Category Id',
    'Benefit per order',
    'Product Status',
    'Customer City',
    'Order City',
    'Order Country',
    'Order State',
    'Customer State',
    'Market'
]
## remove the all coloumns that are not using this code
df.drop(columns=columns_to_drop, inplace=True, errors='ignore')
## remove the canceleing the order they are not relevent
df = df[df['Order Status'] != 'Shipping Canceled']

In [ ]:
## DATA CONVERSION
print('/nduplicate value')
print(df.duplicated().sum())   ## duplicate value check
print(df.isna().sum().sort_values(ascending=False).head(20))

In [ ]:
df.shape


In [ ]:
columns_to_drop = [
    'Customer Id',
    'Order Customer Id',
    'Order Id',
    'Order Item Discount'  ,
    'Order Item Discount Rate']
## remove the all coloumns that are not using this code
df.drop(columns=columns_to_drop, inplace=True, errors='ignore')

In [ ]:
## standard date convertion
for c in ['order date (DateOrders)' , 'shipping date (DateOrders)']:
    df[c] = pd.to_datetime(df[c] , errors='coerce')

In [ ]:
##values counts for categorical columns with low cardinality
for col in df.columns:
  if df[col].nunique() < 10:
    print(col)
    print(df[col].value_counts())

**HOW TO SOLVE THE BUSINESS PROBLEM>>>>**

In [ ]:
##calculating oredr processing time and delay
df['Order Processing Time'] = (df['shipping date (DateOrders)'] - df['order date (DateOrders)']).dt.days
df['Delay'] = df['Order Processing Time'] - df['Days for shipment (scheduled)']
df['Is_Delayed'] = df['Delay']> 0
df['order_month'] = df['order date (DateOrders)'].dt.month
df['order_day'] = df['order date (DateOrders)'].dt.day
df['order_hour'] = df['order date (DateOrders)'].dt.hour
df.describe()

In [ ]:
df['Is_Delayed'].value_counts()

In [ ]:
df.columns.sort_values()

In [ ]:
df['Order Profit Per Order'] >0

In [ ]:
##PROFITABLITY BASED ON ORDER PROFIT PER ORDER
df['Profitablity Flag'] = np.where(df['Order Profit Per Order'] > 0, 'Profitable', np.where(df['Order Profit Per Order'] < 0 ,'loss','Not Profitable'))
df['Profitablity Flag'].value_counts()

In [ ]:

##VISULIZATION OF PROFIT STRUCTURE
Profit_counts = df['Profitablity Flag'].value_counts(normalize=True) * 100
Profit_counts.plot(kind='pie', autopct='%1.1f%%' , colors=[accent_color,danger_color,secondary_color])
plt.ylabel('')
plt.title('Profit Distribution ')
plt.show()

In [ ]:
df['Profitablity Flag'].value_counts(normalize=True)

In [ ]:
def format_func(mom):
  if mom >= 1e6:
    return f'{mom / 1e6:.1f}M $'
  elif mom >= 1e3:
    return f'{mom / 1e3:.1f}K $'
  else:
    return f'{mom} $'

In [ ]:
delayed_df = df[df['Delay']>0]
metrics = {}
metrics['Total Orders'] = len(df)
metrics['Late Deliveries'] = len(delayed_df)
metrics['90% Delay (days)'] = delayed_df['Delay'].quantile(0.9)
metrics['On Time Delivery %'] = (1-float(metrics['Late Deliveries'])/metrics['Total Orders'])*100
metrics['Late Delivery %'] = float(metrics['Late Deliveries'])/metrics['Total Orders']*100
metrics['Total Profit'] = format_func(df.loc[df['Order Profit Per Order']>0 , 'Order Profit Per Order'].sum())
metrics['Total Loss due to delays'] = format_func(df.loc[df['Delay']>0 , 'Order Profit Per Order'].sum())

print('Business KPIS')
for k,v in metrics.items():
  if isinstance(v,float):
     print(f'{k}: {v}')
  else:
    print(f'{k}: {v}')

**Profitability vs Delivery Time Analysis**

In [ ]:
Profit_metrics = (
    df.groupby('Delay')['Order Profit Per Order']
    .agg(mean_profit = 'mean',
         total_Profit ='sum',
         order_count = 'count'
         )
    .reset_index()
)

In [ ]:
Profit_metrics

In [ ]:
delay_distribution = (
    df['Delay'].value_counts(normalize=True)
    .sort_index() *100
).reset_index()

In [ ]:
delay_distribution

In [ ]:
from collections import deque
delay_distribution.columns = ['Delay_Days' ,'Percentage']

print("Profit Metrics by Delay Day:")
print(Profit_metrics.round(1))

print("Delay Distribution:")
display(delay_distribution)

fig,(ax1,ax2) = plt.subplots(1,2,figsize=(12,5))


##first subplot :delay distribution
sns.barplot(x='Delay_Days',y='Percentage', data=delay_distribution , color = accent_color , ax=ax1)
ax1.set_title('Delay Distribution')
ax1.set_xlabel('Delay Days')
ax1.set_ylabel('Percentage')
ax1.set_ylabel('percentage of order(%)')


## percentage text on bars
for bar in ax1.patches:
  height = bar.get_height()
  ax1.text(bar.get_x() + bar.get_width()/2,height+0.5,f'{height:.1f}%',ha='center',va='bottom')


## Second Subplot : profit analysis by delay days
ax2.set_ylabel('total profit', color = primary_color)
ax2.bar(Profit_metrics['Delay'],Profit_metrics['total_Profit'],color=primary_color,label='Total Profit')
ax2.tick_params(axis='y',labelcolor=primary_color)

ax3 = ax2.twinx()

ax3.set_xlabel('Delays Days')
ax3.set_ylabel('Mean Profit',color=secondary_color)
ax3.plot(Profit_metrics['Delay'], Profit_metrics['mean_profit'], marker='o', label="mean profit" , color = accent_color)
ax3.tick_params(axis='y',labelcolor=accent_color)


##format total  profit axis to k $ , m $

def format_func(value , tick_number):
  if value>=1e6:
    return f'{value/1e6:.1f}M $'
  elif value >=1e3:
    return f'{value/1e3:.1f}K $'
  else:
    return f'{value:.0f} $'

ax2.yaxis.set_major_formatter(ticker.FuncFormatter(format_func))
ax3.set_title('profit analysis by delay days')

lines , labels = ax3.get_legend_handles_labels()
lines2 , labels2 = ax2.get_legend_handles_labels()
ax3.legend(lines + lines2 , labels + labels2 , loc ='upper right')
ax3.grid(True , linestyle=':',alpha =0.5)

plt.tight_layout()
plt.show()

**BOTTLENECK DETECTION**

In [ ]:
def compute_delay_pct_by_category(category):
  cat_df = df.groupby(category).agg(
    total_orders = ('Delay','count'),
    late_order = ('Is_Delayed','sum')
  ).reset_index()
  cat_df['delay_pct'] = cat_df['late_order']/cat_df['total_orders']*100
  cat_df = cat_df.sort_values('delay_pct',ascending=False).head(10)
  return cat_df

categories = ['Order Region','Customer Segment','Shipping Mode','Order Status','Type','Department Name']
fig , axes = plt.subplots(2,3,figsize=(15,10),constrained_layout = True)
axes = axes.flatten()


for ax , category in zip (axes , categories):
  cat_df = compute_delay_pct_by_category(category)
  sns.barplot(data = cat_df , x='delay_pct',y=category,ax=ax , palette='viridis')
  ax.set_title(f'Delay % by {category}')
  ax.set_xlabel('')
  ax.set_ylabel(category)
  for i , row in cat_df.reset_index().iterrows():
    ax.text(row['delay_pct']-15,i , f"{row['delay_pct']:.1f}%", va='center', fontsize=10 , color = 'white')
plt.show()

In [ ]:
def compute_delay_pct_by_category(category):
  cat_df = df.groupby(category).agg(
    total_orders = ('Delay','count'),
    late_order = ('Is_Delayed','sum')
  ).reset_index()
  cat_df['delay_pct'] = cat_df['late_order']/cat_df['total_orders']*100
  cat_df = cat_df.sort_values('delay_pct',ascending=False).head(10)
  return cat_df

In [ ]:
compute_delay_pct_by_category('Customer Segment')

**ROOT CAUSE ANALYSIS**

In [ ]:
## TOP DRIVERS OF LATE DELIVERY BY REGION

def top_drivers_by_region(region):
  region_df = df[df['Order Region'] == region].copy()
  drivers = ['Shipping Mode','Customer Segment','Department Name','Type','Order Status'] # Corrected typos
  all_factors = []
  for factor in drivers:
    temp = (
        region_df.groupby(factor).agg( # Corrected df_region to region_df
            total_orders = ('Delay','count'),
            late_orders = ('Is_Delayed','sum'),
            avg_delay = ('Delay','mean')
        ).reset_index()
    )
    temp['delay_pct'] = temp['late_orders']/temp['total_orders']*100
    temp['Driver'] = factor
    temp['Factor_Level'] = factor + ' - ' + temp[factor].astype(str)

    all_factors.append(temp[['Driver','Factor_Level','delay_pct','avg_delay','total_orders']])

##combine all drivers
  final_df = pd.concat(all_factors)

  ## top 10 drivers
  top_factors = final_df.sort_values('delay_pct',ascending=False).head(10)
  plt.figure()

  bars = plt.barh(top_factors['Factor_Level'], top_factors['delay_pct'])

  plt.xlabel("Delay Percentage(%)") # Corrected xlable to xlabel
  plt.ylabel("Driver Factors") # Corrected ylable to ylabel
  plt.title(f"Top Drivers of late delivery in {region}") # Corrected deleviry to delivery
  plt.gca().invert_yaxis()
  for bar in bars:
    width = bar.get_width()
    plt.text(width - 10 , bar.get_y() + bar.get_height()/2,
             f"{width:.1f}%",
             va='center' , fontsize = 10, color = 'white') # Corrected forntsize to fontsize
  plt.show()
top_drivers_by_region('Central Africa')

**TIME BASED ANALYSIS**

In [ ]:
df['shipping date (DateOrders)'].max()

In [ ]:
## delay % by month day of week , hour

delay_by_month = (
    df.groupby('order_month')['Is_Delayed'].mean().reset_index())
delay_by_month['delay_pct'] = delay_by_month['Is_Delayed']*100
delay_by_day = (
    df.groupby('order_day')['Is_Delayed'].mean().reset_index())
delay_by_day['delay_pct'] = delay_by_day['Is_Delayed']*100
delay_by_hour = (
    df.groupby('order_hour')['Is_Delayed'].mean().reset_index())
delay_by_hour['delay_pct'] = delay_by_hour['Is_Delayed']*100

In [ ]:
delay_by_day
# delay_by_hour
# delay_by_month

In [ ]:
from numpy._core.defchararray import center
from pandas.tseries.offsets import Day
fig , (ax1,ax2,ax3) = plt.subplots(1,3,figsize=(15,5),constrained_layout=True)

##  subplot 1: delay % trend over month
ax1.plot(delay_by_month['order_month'],delay_by_month['delay_pct'],marker='o',color=accent_color)
ax1.set_xticks(range(1,13))
ax1.set_xticklabels(['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec'],rotation=45)
ax1.set_xlabel('Month')
ax1.set_ylabel('Delay %')
ax1.set_title('delay % trend over month')
ax1.grid(True , linestyle=':',alpha=0.5)

## Annotate top 3 highest
top3_month = delay_by_month.nlargest(3,'delay_pct')
for _ , row in top3_month.iterrows():
  ax1.annotate(f"{row['delay_pct']:.1f}%",(row['order_month'],row['delay_pct']),
               textcoords = "offset points" , xytext = (0,10),
               ha = 'center',fontsize=10,color = danger_color)

##subplot 2 : delay % by day of week
Day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
# Create 'order_day_of_week' column by mapping 'order_day' to day names
df['order_day_of_week'] = df['order date (DateOrders)'].dt.day_name()
delay_by_day_of_week = df.groupby('order_day_of_week')['Is_Delayed'].mean().reindex(Day_order).reset_index()
delay_by_day_of_week['delay_pct'] = delay_by_day_of_week['Is_Delayed']*100

ax2.bar(delay_by_day_of_week['order_day_of_week'],delay_by_day_of_week['delay_pct'],color = primary_color)
ax2.set_xticklabels(delay_by_day_of_week['order_day_of_week'],rotation=45)
ax2.set_xlabel('Day of Week')
ax2.set_ylabel('Delay %')
ax2.set_title('delay % by day of week')
ax2.grid(True , linestyle=':',alpha=0.5)

##Annotate top 3 highest bars
top3_day = delay_by_day_of_week.nlargest(3,'delay_pct')
for _ , row in top3_day.iterrows():
     height = row['delay_pct']
     ax2.text(row['order_day_of_week'] , height + 0.5, f'{height:.1f}%',ha='center',va='bottom',color='black',fontsize=10)

##subplot 3 : Delay % by hour
ax3.plot(delay_by_hour['order_hour'],delay_by_hour['delay_pct'],marker='o',color=primary_color)
ax3.set_xlabel('Hour')
ax3.set_ylabel('Delay %')
ax3.set_title('delay % by hour')
ax3.grid(True , linestyle=':',alpha=0.5)


##annotate top 3 highest
top3_hour = delay_by_hour.nlargest(3,'delay_pct')
for _ , row in top3_hour.iterrows():
  ax3.annotate(f"{row['delay_pct']:.1f}%",(row['order_hour'],row['delay_pct']),
  textcoords = "offset points" , xytext = (0,10), ha='center',fontsize=10,color = danger_color)

plt.tight_layout()
plt.show()

**MACHINELEARNING MODEL FITTTING**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score , classification_report
from sklearn.ensemble import RandomForestClassifier
from collections import Counter
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler

In [ ]:
x = df[['Type','Days for shipment (scheduled)','Category Name','Customer Segment','Department Name','Order Region','Shipping Mode','order_month','order_hour']]
y = df['Late_delivery_risk']

In [ ]:
cat_cols = x.select_dtypes(include=['object','category']).columns.tolist()
print('Categorical columns:', cat_cols)

## Frequency encoding ()

for col in cat_cols:
  freq = x[col].value_counts(normalize=True)
  x[f'{col}_freq'] = x[col].map(freq)

## keep added numeric columns and encoded features , drop original string categories
x_encoded = x.drop(columns=cat_cols)
print('Shape after freq+target encoding:',x_encoded.shape)

# use encoded features

x=x_encoded


## train/test split after encoding
x_train , x_test , y_train , y_test = train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

In [ ]:
y.value_counts()

In [ ]:

## balancing the traning data using SMOTE
print('Before balancing (train):',Counter(y_train))

smote = SMOTE(random_state=42)
x_train_bal , y_train_bal = smote.fit_resample(x_train,y_train)
print("After balancing (train):",Counter(y_train_bal))

In [ ]:
def evaluate_model(y_true , y_pred, model_name="Model"):
  accuracy = accuracy_score(y_true,y_pred)
  precision = precision_score(y_true,y_pred)
  recall = recall_score(y_true,y_pred)
  f1 = f1_score(y_true,y_pred)

  print(f"\n--- {model_name} Evaluation ---")
  print(f"Accuracy: {accuracy:.4f}")
  print(f"Precision: {precision:.4f}")
  print(f"Recall: {recall:.4f}")
  print(f"F1-Score: {f1:.4f}")
  print("\nClassification Report:")
  print(classification_report(y_true,y_pred))

In [ ]:
rf_model_balanced = RandomForestClassifier(random_state=42)
rf_model_balanced.fit(x_train_bal,y_train_bal)

y_pred_balanced = rf_model_balanced.predict(x_test)
evaluate_model(y_test,y_pred_balanced,'Random Forest Classifier')